# BHARATH BUILDS
## 21 Days to ML Engineer

# Day 10 — Model Selection, Cross-Validation & Hyperparameter Tuning

**Today's Topic:** Model Selection + Cross-Validation + Hyperparameter Tuning

## 🎯 Objectives of Today

The objective of today's session is to understand how to compare different machine learning models, evaluate their performance using cross-validation, and select suitable hyperparameters to build a reliable and reproducible machine learning workflow.

By the end of this session, we will understand how to select a suitable model, prevent data leakage, and evaluate the final model on unseen data.

## 📚 Theoretical Aspects

Today, we will cover the following concepts:

1. **Training, Validation and Test Sets** — Understanding the purpose of each dataset split.
2. **K-Fold Cross-Validation** — Evaluating a model across multiple folds.
3. **Stratified K-Fold Cross-Validation** — Maintaining approximately the same class proportions across classification folds.
4. **Parameters vs Hyperparameters** — Understanding learned values and configurable settings.
5. **Hyperparameter Tuning** — Finding suitable model configurations.
6. **GridSearchCV** — Testing every specified hyperparameter combination.
7. **RandomizedSearchCV** — Evaluating sampled hyperparameter configurations.
8. **Data Leakage** — Understanding how leakage can produce misleading evaluation results.
9. **Pipeline-Based Modeling** — Combining preprocessing and model fitting into one workflow.
10. **Fair Model Comparison** — Comparing models using a consistent evaluation strategy.

## 💻 Practical Aspects — Google Colab

We will implement a complete model-selection experiment using the **Breast Cancer Wisconsin dataset** from scikit-learn.

### Step 1: Import Libraries
Import the required Python and scikit-learn libraries.

### Step 2: Load the Dataset
Load the dataset and separate input features (X) and target (y).

### Step 3: Train/Test Split
Create an 80/20 stratified train/test split and keep the test set untouched during model selection.

### Step 4: Build Model Pipelines
Create Pipelines for five classification models:
- Logistic Regression
- K-Nearest Neighbors (KNN)
- Decision Tree
- Support Vector Machine (SVM)
- Random Forest

### Step 5: Configure Cross-Validation
Set up five-fold Stratified K-Fold Cross-Validation.

### Step 6: Compare Models
Evaluate all five models using the same cross-validation strategy and ROC-AUC scoring metric.

### Step 7: Hyperparameter Tuning
Use GridSearchCV to tune the SVM hyperparameters and identify the best configuration.

### Step 8: Final Test Evaluation
Evaluate the selected SVM model using the untouched test set.

Calculate:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

### Step 9: Save Results
Save the model comparison results, best hyperparameters, test metrics, and fitted model for reproducibility.

### Step 10: Interpret and Conclude
Compare the results, explain the observations, and summarize the complete model-selection workflow.

## 🔄 Machine Learning Pipeline

```text
         Load Dataset
               |
               v
        Train/Test Split
               |
               v
        Define Model Pipelines
               |
               v
     Stratified K-Fold CV
               |
               v
        Compare 5 Models
               |
               v
      Hyperparameter Tuning
          (GridSearchCV)
               |
               v
      Select Best Configuration
               |
               v
      Final Test Evaluation
               |
               v
        Save Results
```

## 🛠️ Tools and Technologies

- Python
- Google Colab
- Pandas
- Scikit-learn
- GridSearchCV
- StratifiedKFold
- Pipeline
- Joblib

## 🎓 Expected Learning Outcomes

By completing today's practical session, we will be able to:

- Compare multiple classification models fairly.
- Explain and implement cross-validation.
- Tune hyperparameters systematically.
- Prevent common preprocessing leakage.
- Evaluate a selected model using an untouched test set.
- Save experiment results for reproducibility.

## 🔑 Key Principle

**EXPLAIN → IMPLEMENT → INTERPRET → DOCUMENT → COMMIT**

**Bharath Builds in AI — Learn. Build. Explain.**


step 1: import libraries

In [1]:
# Step 1: Import Libraries

import json
import joblib
import pandas as pd

from pathlib import Path

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully!")

All libraries imported successfully!


step 2: load the dataset

In [3]:
# Step 2: Load the Dataset

cancer = load_breast_cancer()

X = pd.DataFrame(
    cancer.data,
    columns=cancer.feature_names
)

y = pd.Series(
    cancer.target,
    name="target"
)

print("Dataset shape:", X.shape)
print("Number of features:", X.shape[1])

print("\nTarget distribution:")
print(
    y.map({0: "malignant", 1: "benign"})
     .value_counts()
)

Dataset shape: (569, 30)
Number of features: 30

Target distribution:
target
benign       357
malignant    212
Name: count, dtype: int64


step 3: Train/test split


In [4]:
# Step 3: Stratified Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Test samples:", X_test.shape[0])

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).round(3))

Training samples: 455
Test samples: 114

Training class proportions:
target
1    0.626
0    0.374
Name: proportion, dtype: float64

Test class proportions:
target
1    0.632
0    0.368
Name: proportion, dtype: float64


step 4: Define five models with pipelines

In [7]:
# Step 4: Define Model Pipelines

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier())
    ]),

    "Decision Tree": Pipeline([
        ("classifier", DecisionTreeClassifier(
            random_state=42
        ))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            probability=True,
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=1
        ))
    ])
}

print("Models created successfully:\n")

for name in models:
    print("-", name)

Models created successfully:

- Logistic Regression
- KNN
- Decision Tree
- SVM
- Random Forest


Step 5 — Configure Stratified K-Fold Cross-Validation

In [9]:
# Step 5: Configure Stratified K-Fold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation strategy:", cv)
print("Number of folds:", cv.get_n_splits())

Cross-validation strategy: StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Number of folds: 5


step 6: compare the five models

In [10]:
# Step 6: Compare Models Using Cross-Validation

results = []

for name, pipeline in models.items():

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "Mean CV ROC-AUC": scores.mean(),
        "Std CV ROC-AUC": scores.std()
    })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Mean CV ROC-AUC",
    ascending=False
).reset_index(drop=True)

display(results_df.round(4))

,Model,Mean CV ROC-AUC,Std CV ROC-AUC
0,Logistic Regression,0.9959,0.0050
1,SVM,0.9956,0.0048
2,Random Forest,0.9896,0.0083
3,KNN,0.9882,0.0081
4,Decision Tree,0.9179,0.0201


Step 7 — Hyperparameter Tuning Using GridSearchCV

In [11]:
# Step 7: Tune SVM Using GridSearchCV

svm_pipeline = models["SVM"]

param_grid = {
    "classifier__C": [0.1, 1, 10],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto"]
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print("Best SVM parameters:")
print(grid_search.best_params_)

print("\nBest mean CV ROC-AUC:")
print(round(grid_search.best_score_, 4))

Best SVM parameters:
{'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}

Best mean CV ROC-AUC:
0.9957


Step 8 — Final Test Evaluation

In [12]:
# Step 8: Evaluate the Tuned SVM on the Test Set

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

print("Final Test Metrics:")

for metric, score in test_metrics.items():
    print(f"{metric}: {score:.4f}")

print("\nClassification Report:")

print(classification_report(
    y_test,
    y_pred,
    target_names=cancer.target_names
))

print("Confusion Matrix:")

print(confusion_matrix(y_test, y_pred))

Final Test Metrics:
accuracy: 0.9825
precision: 0.9861
recall: 0.9861
f1_score: 0.9861
roc_auc: 0.9937

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

Confusion Matrix:
[[41  1]
 [ 1 71]]


Step 9 — Save Experiment Results

In [13]:
# Step 9: Save Experiment Results

output_dir = Path("day10_results")
output_dir.mkdir(exist_ok=True)

# Save baseline model comparison
results_df.to_csv(
    output_dir / "cv_results.csv",
    index=False
)

# Save all SVM grid-search results
pd.DataFrame(
    grid_search.cv_results_
).to_csv(
    output_dir / "svm_grid_search_results.csv",
    index=False
)

# Save the best configuration
best_config = {
    "selected_model": "SVM",
    "best_params": grid_search.best_params_,
    "best_mean_cv_roc_auc": float(grid_search.best_score_),
    "cv_folds": 5,
    "random_state": 42,
    "scoring_metric": "roc_auc"
}

with open(
    output_dir / "best_model_config.json",
    "w"
) as file:
    json.dump(best_config, file, indent=4)

# Save final test metrics
with open(
    output_dir / "test_metrics.json",
    "w"
) as file:
    json.dump(
        {k: float(v) for k, v in test_metrics.items()},
        file,
        indent=4
    )

# Save the fitted SVM Pipeline
joblib.dump(
    best_model,
    output_dir / "best_svm_pipeline.joblib"
)

print("Saved experiment files:")

for path in sorted(output_dir.iterdir()):
    print(path)

Saved experiment files:
day10_results/best_model_config.json
day10_results/best_svm_pipeline.joblib
day10_results/cv_results.csv
day10_results/svm_grid_search_results.csv
day10_results/test_metrics.json


Step 10 — Final Summary and Conclusion

In [14]:
# Step 10: Final Summary

print("=" * 50)
print("DAY 10: MODEL SELECTION SUMMARY")
print("=" * 50)

print("\nModels compared:", len(models))
print("Cross-validation folds:", cv.get_n_splits())
print("Comparison metric: ROC-AUC")

print("\nTop baseline model:")
print(results_df.iloc[0]["Model"])

print("\nTuned model: SVM")
print("Best parameters:", grid_search.best_params_)

print(
    "Best CV ROC-AUC:",
    round(grid_search.best_score_, 4)
)

print(
    "Final test ROC-AUC:",
    round(test_metrics["roc_auc"], 4)
)

print("\nResults saved in:", output_dir)

print("\nDay 10 experiment completed successfully!")

DAY 10: MODEL SELECTION SUMMARY

Models compared: 5
Cross-validation folds: 5
Comparison metric: ROC-AUC

Top baseline model:
Logistic Regression

Tuned model: SVM
Best parameters: {'classifier__C': 0.1, 'classifier__gamma': 'scale', 'classifier__kernel': 'linear'}
Best CV ROC-AUC: 0.9957
Final test ROC-AUC: 0.9937

Results saved in: day10_results

Day 10 experiment completed successfully!


##  Step 11 — Model Comparison: Interpretation of Results

After running five-fold Stratified Cross-Validation, we obtained the following results.

### 1. Logistic Regression — Best Baseline Model

- Mean CV ROC-AUC: **0.9959**
- Standard deviation: **0.0050**
- Logistic Regression achieved the highest mean ROC-AUC among the five baseline models.
- This indicates excellent ranking performance across the validation folds.
- It is a strong baseline for this classification problem.

### 2. SVM — Very Close to Logistic Regression

- Mean CV ROC-AUC: **0.9956**
- Standard deviation: **0.0048**
- SVM performed almost as well as Logistic Regression.
- Its fold-to-fold variation was slightly lower than Logistic Regression's.
- The difference between their mean ROC-AUC scores is very small.

### 3. Random Forest — Strong Performance

- Mean CV ROC-AUC: **0.9896**
- Standard deviation: **0.0083**
- Random Forest achieved strong validation performance.
- Its mean score was lower than Logistic Regression and SVM.
- Its results varied somewhat more across folds than those of the top two models.

### 4. K-Nearest Neighbors (KNN) — Good Performance

- Mean CV ROC-AUC: **0.9882**
- Standard deviation: **0.0081**
- KNN performed well, but its mean score was lower than the top three models.
- Feature scaling was included because KNN relies on distances between samples.

### 5. Decision Tree — Lowest Mean Score

- Mean CV ROC-AUC: **0.9179**
- Standard deviation: **0.0201**
- Decision Tree achieved the lowest mean ROC-AUC among the five baseline models.
- It also showed the greatest variation across folds.
- This suggests that its performance was less consistent in this experiment.

##  Step 12 — What Do These Results Tell Us?

1. **Logistic Regression ranked first among the baseline models.** It achieved a mean CV ROC-AUC of approximately 0.9959.

2. **SVM was a very close second.** Its mean CV ROC-AUC was approximately 0.9956.

3. **Random Forest and KNN also performed strongly.** Their mean CV ROC-AUC scores were approximately 0.9896 and 0.9882, respectively.

4. **Decision Tree performed less consistently.** It had both the lowest mean score and the highest standard deviation among the five models.

5. **The top two models were very close.** We should not claim that Logistic Regression is meaningfully better than SVM based on this small difference alone.

6. **Cross-validation helped us compare models fairly.** Every baseline model used the same training data, five-fold strategy, and ROC-AUC scoring metric.

##  Step 13 — SVM Hyperparameter Tuning Results

The GridSearchCV experiment selected the following configuration:

- Model: SVM
- C: 0.1
- Kernel: Linear
- Gamma: `scale`
- Best mean CV ROC-AUC: **0.9957**
- Final test ROC-AUC: **0.9937**

### Interpretation

- GridSearchCV selected the best configuration from the specified hyperparameter combinations.
- The tuned SVM achieved a mean CV ROC-AUC of approximately 0.9957.
- Its final test ROC-AUC was approximately 0.9937, indicating excellent ranking performance on the held-out test set.
- The test score is slightly lower than the best CV score, which is a normal possible outcome.
- We should not conclude that SVM is the overall winner just because it was tuned. Logistic Regression achieved a slightly higher mean CV ROC-AUC in the baseline comparison.

**Important observation:** The selected kernel was linear, suggesting that the linear SVM configuration performed best among the candidates evaluated by this search. It does not prove that a linear boundary is always best for this dataset or for other datasets.

### Final Takeaway

**There is no universally best machine learning algorithm.** The suitable model depends on the dataset, evaluation metric, generalization performance, computational cost, and requirements of the problem.

In this experiment, Logistic Regression was the strongest baseline model, while the tuned SVM also achieved excellent performance.


